# MICrONS: Structure–Function Pair Analysis

The MICrONS program links cellular-resolution electron-microscopy reconstruction with in vivo visual physiology. This notebook illustrates a pairwise analysis: compare a structural connection indicator or synapse count with a functional similarity measure. The default data are synthetic, preserving the analysis workflow while avoiding credentials, large downloads, and release-specific table assumptions.

Association between a reconstructed connection and response similarity is not proof that the synapse causes tuning, nor does it establish direction of functional influence. Pairwise observations are also statistically dependent because cells recur across pairs.

## Quick start

For the default offline path, run the minimal-install cell and then run the synthetic pair-analysis cells from top to bottom. You should see connected-versus-unconnected response-similarity distributions, a permutation null, and a synthetic synapse-count association. The later CAVE cells are optional advanced previews and do not supply data to or alter the synthetic analysis.

## Prerequisites

- Python 3.9+, NumPy/Pandas, and basic correlation and permutation-test concepts
- Familiarity with calcium imaging, electron microscopy, and the distinction between a synapse count and synaptic efficacy
- Optional live exploration requires a CAVE account/token and authorization for the selected materialization

## Setup

Run this minimal installation for the default offline tutorial.

In [ ]:
%pip install -q numpy pandas scipy matplotlib

### Optional installation for advanced live CAVE previews

Install this client only if you will enable the advanced preview cells below. The default synthetic analysis does not import it.

In [ ]:
%pip install -q caveclient

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

rng = np.random.default_rng(17)

## Advanced preview (optional): connect to CAVE

CAVE tables and materialization versions are release-specific. Enable this cell only with an authorized `CAVE_TOKEN`, then inspect the available tables and schema before writing a query. Do not assume that a table name, segmentation root ID, or coordinate frame from one MICrONS release applies to another.

In [ ]:
USE_LIVE_CAVE = False
CAVE_DATASTACK = 'minnie65_public_v117'  # verify this against the current CAVE documentation

if USE_LIVE_CAVE:
    from caveclient import CAVEclient
    if not os.environ.get('CAVE_TOKEN'):
        raise RuntimeError('Set CAVE_TOKEN before enabling live CAVE access.')
    client = CAVEclient(CAVE_DATASTACK, auth_token=os.environ['CAVE_TOKEN'])
    print(client.materialize.get_tables()[:10])
else:
    print('Live CAVE access is disabled; using a synthetic structure-function dataset.')


## Advanced preview (optional): inspect and query one verified CAVE materialization table

This advanced preview does not feed the synthetic analysis below. CAVE schemas are datastack- and materialization-specific, so this cell intentionally supplies no table name, root ID, or root-ID column. With `USE_LIVE_CAVE=True`, inspect `get_tables()` and `get_table_metadata()` first. Then set all three `LIVE_*` values from the returned schema and your own verified root ID. The query is capped at 10 rows and uses `filter_in_dict`, avoiding a broad table download.

In [ ]:
QUERY_LIVE_CAVE = False
LIVE_TABLE_NAME = None      # exact name selected from client.materialize.get_tables()
LIVE_ROOT_ID_COLUMN = None  # exact root-ID column verified in that table's metadata/schema
LIVE_ROOT_ID = None         # integer root ID verified for this datastack/materialization
LIVE_LIMIT = 10

if QUERY_LIVE_CAVE:
    if not USE_LIVE_CAVE or 'client' not in globals():
        raise RuntimeError('Enable and run the CAVE connection cell first.')
    if not isinstance(LIVE_TABLE_NAME, str) or not LIVE_TABLE_NAME:
        raise ValueError('Set LIVE_TABLE_NAME after inspecting client.materialize.get_tables().')
    if not isinstance(LIVE_ROOT_ID_COLUMN, str) or not LIVE_ROOT_ID_COLUMN:
        raise ValueError('Set LIVE_ROOT_ID_COLUMN from verified table metadata/schema.')
    if not isinstance(LIVE_ROOT_ID, (int, np.integer)):
        raise ValueError('Set LIVE_ROOT_ID to a verified integer root ID.')

    table_metadata = client.materialize.get_table_metadata(LIVE_TABLE_NAME)
    print('Table metadata:', table_metadata)
    live_rows = client.materialize.query_table(
        LIVE_TABLE_NAME,
        filter_in_dict={LIVE_ROOT_ID_COLUMN: [LIVE_ROOT_ID]},
        limit=LIVE_LIMIT,
    )
    display(live_rows)
else:
    print('Live CAVE table query is disabled; no materialization rows will be downloaded.')


## Generate response features and directed structural pairs

We simulate orientation-like response vectors for 40 cells. A directed pair is assigned a synthetic synapse count partly related to its tuning similarity. This built-in relationship makes the code demonstrable—it must not be interpreted as an estimate of the MICrONS effect size.

In [ ]:
n_cells, n_stimuli = 40, 12
cell_ids = np.arange(10_000, 10_000 + n_cells)
preferred = rng.uniform(0, np.pi, n_cells)
orientations = np.linspace(0, np.pi, n_stimuli, endpoint=False)
responses = np.array([np.cos(2 * (orientations - pref)) + rng.normal(0, 0.25, n_stimuli) for pref in preferred])
responses = (responses - responses.mean(axis=1, keepdims=True)) / responses.std(axis=1, keepdims=True)

rows = []
for i in range(n_cells):
    for j in range(n_cells):
        if i == j:
            continue
        similarity = np.corrcoef(responses[i], responses[j])[0, 1]
        synapses = rng.poisson(np.exp(-0.3 + 0.9 * similarity))
        rows.append((cell_ids[i], cell_ids[j], similarity, synapses))
pairs = pd.DataFrame(rows, columns=['pre_root_id', 'post_root_id', 'response_similarity', 'synapse_count'])
pairs['connected'] = pairs['synapse_count'] > 0
pairs.head()

## Compare connected and unconnected pairs

The descriptive comparison uses all ordered pairs. It is useful for visualization, but its uncertainty is too optimistic if pairs are treated as independent. We next use a cell-label permutation test that preserves each cell's response vector while disrupting the alignment between functional features and structural edges.

In [ ]:
summary = pairs.groupby('connected')['response_similarity'].agg(['count', 'mean', 'median'])
print(summary)

connected_values = pairs.loc[pairs.connected, 'response_similarity']
unconnected_values = pairs.loc[~pairs.connected, 'response_similarity']
plt.hist(unconnected_values, bins=25, alpha=0.6, label='no synthetic synapse')
plt.hist(connected_values, bins=25, alpha=0.6, label='≥1 synthetic synapse')
plt.xlabel('Response-vector correlation'); plt.ylabel('Ordered pairs'); plt.legend();
plt.title('Synthetic pairwise structure–function comparison')

In [ ]:
def connected_minus_unconnected(frame):
    return (frame.loc[frame.connected, 'response_similarity'].mean()
            - frame.loc[~frame.connected, 'response_similarity'].mean())

observed = connected_minus_unconnected(pairs)
null = []
for _ in range(1_000):
    shuffled = pairs.copy()
    shuffled['response_similarity'] = rng.permutation(shuffled['response_similarity'].to_numpy())
    null.append(connected_minus_unconnected(shuffled))
null = np.asarray(null)
p_value = (1 + np.sum(np.abs(null) >= abs(observed))) / (len(null) + 1)
print(f'Observed mean difference: {observed:.3f}')
print(f'Two-sided label-permutation p-value: {p_value:.4f}')

plt.hist(null, bins=35, color='lightgray', label='permuted alignment')
plt.axvline(observed, color='tab:red', label='observed')
plt.xlabel('Connected − unconnected mean similarity'); plt.legend();

## Synapse-count association and cautions

A rank correlation summarizes monotonic association among connected pairs. In a real analysis, additionally control or stratify for distance, cell class, cortical layer, reconstruction completeness, activity quality, and experiment/session. Use a hierarchical or node-resampling approach for valid uncertainty because source and target neurons appear in many pairs.

In [ ]:
connected = pairs.query('synapse_count > 0')
rho, p_naive = spearmanr(connected['synapse_count'], connected['response_similarity'])
print(f'Spearman rho (synthetic connected pairs): {rho:.3f}; naive p={p_naive:.3g}')
print('Naive p treats pairs as independent; do not report it as final inferential evidence.')

plt.scatter(connected['synapse_count'], connected['response_similarity'], alpha=0.45)
plt.xlabel('Synthetic synapse count'); plt.ylabel('Response-vector correlation')
plt.title('Structural weight versus functional similarity')

## References

- MICrONS Consortium et al. (2021). Functional connectomics spanning multiple areas of mouse visual cortex. *bioRxiv*. https://doi.org/10.1101/2021.07.28.454025
- Turner, N. L., Macrina, T., Bae, J. A., et al. (2022). Reconstruction of neocortex: Organelles, compartments, cells, circuits, and activity. *Cell*, 185(6), 1082–1100.e24. https://doi.org/10.1016/j.cell.2022.01.023
- CAVEclient documentation. https://caveclient.readthedocs.io/
- MICrONS Explorer and data access. https://www.microns-explorer.org/

## License

This notebook is released under the repository's license. The generated data are synthetic. Any use of MICrONS data, services, or software must follow the current dataset access terms, licenses, and citation guidance.